# DER-based Baseline Models: Controlled Comparison and Interpretability

**Pipeline:** DER annotations → rule-based relations → per-sentence NetworkX graphs → handcrafted graph-feature baselines and sentence-vector classifiers.

**Main question:** how well can non-neural graph-feature baselines and sentence-only models distinguish stereotype (label 1) from anti-stereotype (label 0), and how sensitive are their predictions to sentence text, DER nodes, relations, and identity substitutions?

This notebook compares frozen SBERT and BERT sentence representations with interpretable feature-based baselines. It does not train a graph neural network. Graph structure is retained only where needed to construct handcrafted baseline features and to run controlled graph-component ablations.

**Research caution:** small datasets can produce unstable estimates. Treat this as an exploratory controlled comparison. Both datasets use stratified CV; on InDiBias, minimal-pair partners can fall in different folds, so read InDiBias scores as optimistic. A label-shuffled LR control is included to check the pipeline for label leakage. Do not select a model solely by its best single score.


## 0. Setup & configuration


In [1]:
# Install dependencies (Colab). spaCy is optional (identity lemmatiser falls back to plural stripping).
%pip install -q transformers sentence-transformers torch-geometric scikit-learn pandas numpy networkx matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import copy, random, re, itertools, json, warnings, gc, time, os, hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt


from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)

# ═════════════════════════ RUN PROFILE ═════════════════════════
RUN_PROFILE = "full"      # "quick": 1 repeat, InDiBias analysis only (sanity pass)
                           # "full" : 3 repeats, both datasets analysed (paper numbers)
PROFILES = {
    "quick": dict(n_repeats=1, abl_repeats=1, n_boot=1000, n_random_del=10,
                  analysis_plan=[("InDiBias", "stratified")]),
    "full":  dict(n_repeats=3, abl_repeats=1, n_boot=5000, n_random_del=20,
                  analysis_plan=[("InDiBias", "stratified"), ("IndiCasa", "stratified")]),
}
P = PROFILES[RUN_PROFILE]

SEED = 42
N_REPEATS   = P["n_repeats"]
ABL_REPEATS = P["abl_repeats"]
N_BOOT      = P["n_boot"]
N_RANDOM_DEL = P["n_random_del"]
SEEDS = [SEED + 100 * r for r in range(N_REPEATS)]
ABL_SEEDS = SEEDS[:ABL_REPEATS]

# Same evaluation plan for every variant: stratified CV on both datasets (no grouped CV).
EVAL_PLAN = [("InDiBias", "stratified"),
             ("IndiCasa", "stratified")]
ANALYSIS_PLAN = P["analysis_plan"]         # plans that also get ablations + interpretability + qualitative
assert all(p in EVAL_PLAN for p in ANALYSIS_PLAN)

# ═════════════════════════ GPU POLICY ═════════════════════════
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_ON_GPU = True        # set False if you want the GPU free for something else
train_device = device if TRAIN_ON_GPU else torch.device("cpu")
ENCODER_FP16 = True        # encoders run in fp16 on CUDA (vectors are cast back to fp32 and cached on CPU)
ENC_BATCH = 64


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_report(tag=""):
    if device.type != "cuda":
        print(f"[gpu] {tag}: running on CPU"); return
    mb = lambda b: b / 2**20
    print(f"[gpu] {tag}: allocated {mb(torch.cuda.memory_allocated()):.0f} MB | "
          f"reserved {mb(torch.cuda.memory_reserved()):.0f} MB | peak {mb(torch.cuda.max_memory_allocated()):.0f} MB")


# Training configs for the sentence-vector MLP baselines
BASE_CFG = dict(epochs=50, lr=1e-3, weight_decay=1e-4, batch_size=16, hidden=64, dropout=0.3)
REG_CFG  = dict(BASE_CFG, hidden=32, dropout=0.5, weight_decay=1e-3, epochs=30)

os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
print(f"profile={RUN_PROFILE} | repeats={N_REPEATS} | seeds={SEEDS} | device={device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
gpu_report("start")


profile=full | repeats=3 | seeds=[42, 142, 242] | device=cpu
[gpu] start: running on CPU


## 1. Data
`pos` = stereotype (label 1), `neg` = anti-stereotype (label 0).
Each InDiBias annotation carries a `template_id`; pos[i] and neg[i] come from the same template (49/50 differ only in the identity term).

In [3]:
import json
from pathlib import Path

DATA_DIR = Path("../data")

with open(DATA_DIR / "indibias_pos.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_POS_INDIBIAS = json.load(f)

with open(DATA_DIR / "indibias_neg.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_NEG_INDIBIAS = json.load(f)

with open(DATA_DIR / "indicasa_pos.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_POS_INDICASA = json.load(f)

with open(DATA_DIR / "indicasa_neg.json", "r", encoding="utf-8") as f:
    EXAMPLE_ANNOTATIONS_NEG_INDICASA = json.load(f)

In [4]:
print("InDiBias positive:", len(EXAMPLE_ANNOTATIONS_POS_INDIBIAS))
print("InDiBias negative:", len(EXAMPLE_ANNOTATIONS_NEG_INDIBIAS))
print("IndiCasa positive:", len(EXAMPLE_ANNOTATIONS_POS_INDICASA))
print("IndiCasa negative:", len(EXAMPLE_ANNOTATIONS_NEG_INDICASA))

InDiBias positive: 243
InDiBias negative: 83
IndiCasa positive: 243
IndiCasa negative: 83


In [5]:
DATASETS = {
    "InDiBias": dict(pos=EXAMPLE_ANNOTATIONS_POS_INDIBIAS, neg=EXAMPLE_ANNOTATIONS_NEG_INDIBIAS),
    "IndiCasa": dict(pos=EXAMPLE_ANNOTATIONS_POS_INDICASA, neg=EXAMPLE_ANNOTATIONS_NEG_INDICASA),
}
for name, d in DATASETS.items():
    print(f"{name}: {len(d['pos'])} stereotype / {len(d['neg'])} anti-stereotype")

InDiBias: 243 stereotype / 83 anti-stereotype
IndiCasa: 243 stereotype / 83 anti-stereotype


## 2. DER annotations → sentence graphs

In [6]:
# Identity normalisation ("Hindus" -> "hindu", "OBC family" -> "obc", ...)
try:
    import spacy
    _nlp = spacy.load("en_core_web_sm")
except (ImportError, OSError):
    _nlp = None          # falls back to simple plural stripping
print("spaCy lemmatiser:", "on" if _nlp is not None else "off (fallback)")

_IDENTITY_MODIFIERS = frozenset({
    "family", "families", "group", "community", "commnunity",
    "people", "person", "man", "woman", "child", "children"
})
_PLURAL_OVERRIDES = {
    "hindus": "hindu", "muslims": "muslim", "vaishyas": "vaishya", "brahmins": "brahmin",
    "dalits": "dalit", "obcs": "obc", "christians": "christian", "sikhs": "sikh",
    "jains": "jain", "buddhists": "buddhist",
}
_COMPOUND_PATTERNS = [
    (r"\b(obc|hindu|muslim|brahmin|dalit|vaishya|christian|sikh|jain)\s+(family|group|community|people)\b", r"\1"),
    (r"\b(hindu|muslim|brahmin|dalit|vaishya)\s+(man|woman|person|child)\b", r"\1"),
]


def _strip_plural(text):
    return text[:-1] if text.endswith("s") and not text.endswith(("ss", "us", "is")) else text


def _normalize_identity(span: str) -> str:
    text = span.lower().strip()
    for pattern, repl in _COMPOUND_PATTERNS:
        text = re.sub(pattern, repl, text, flags=re.IGNORECASE)
    words = text.split()
    if len(words) > 1 and words[-1] in _IDENTITY_MODIFIERS:
        text = " ".join(words[:-1])
    if text in _PLURAL_OVERRIDES:
        text = _PLURAL_OVERRIDES[text]
    if _nlp is not None:
        try:
            text = " ".join(t.lemma_ for t in _nlp(text))
        except Exception:
            text = _strip_plural(text)
    else:
        text = _strip_plural(text)
    return text.strip()

spaCy lemmatiser: on


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# MAPPING IDENTITY TO CATEGORY
# ─────────────────────────────────────────────────────────────────────────────

_IDENTITY_TO_CATEGORY = {
    # Religion
    "hindu": "Religion",
    "muslim": "Religion",
    "christian": "Religion",
    "sikh": "Religion",
    "jain": "Religion",
    "buddhist": "Religion",
    "islam" :  "Religion",
    "christianity": "Religion",
    "hinduism": "Religion",
    "sikhism" : "Religion",
    "parsi" : "Religion",
    "parsis": "Religion",
    "buddhism": "Religion",
    "jainism": "Religion",

    # Caste
    "brahmin": "Caste",
    "vaishya": "Caste",
    "dalit": "Caste",
    "obc": "Caste",
    "sc": "Caste",
    "st": "Caste",
    "kshatriya": "Caste",
    "shudra": "Caste",
    "sc / st": "Caste",
    "kaystha": "Caste",
    "bania": "Caste",
    "kurmi": "Caste",
    "jat": "Caste",
    "rajput": "Caste",
    "adivasi": "Caste",
    "kayast": "Caste",
    "general": "Caste",
    "forward class": "Caste",
    "artisan": "Caste",
    "merchant": "Caste",
    "trader": "Caste",
    "laborer": "Caste",
    "peasant": "Caste",

    # Gender
    "women": "Gender",
    "man": "Gender",
    "woman": "Gender",
    "female": "Gender",
    "male": "Gender",
    "boy": "Gender",
    "girl": "Gender",
    "girls": "Gender",
    "boys": "Gender",
    "men": "Gender",
    "transgender male": "Gender",
    "lesbian": "Gender"
}

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. RELATION EXTRACTION (unchanged structure)
# ══════════════════════════════════════════════════════════════════════════════

class RuleBasedRelations:
    """
    Converts a DER annotation dict into a flat list of canonical edges.
    Structure unchanged, only the node extraction uses new normalization.
    """

    # ── Node-type labels ──────────────────────────────────────────────────
    IDENTITY_NODE   = "Identity Dimensions"
    PROFESSION_NODE = "Profession"
    TRIGGER_NODE    = "Contextual Trigger"
    FRAME_NODE      = "Situational Frame"

    # ── Canonical relation-type vocabulary ────────────────────────────────
    REL_IDENTITY_INTERSECTION        = "identity_intersection"
    REL_IDENTITY_ROLE_ASSOCIATION    = "identity_role_association"
    REL_CHARACTERIZED_AS             = "characterized_as"
    REL_IDENTITY_FRAME_ASSOCIATION= "identity_frame_association"

    # ── Copula / auxiliary / modal verbs ──────────────────────────────────
    PREDICATE_LEAD_WORDS: frozenset = frozenset({
        "is", "are", "was", "were", "be", "been", "being",
        "do", "does", "did",
        "has", "have", "had",
        "can", "could", "will", "would", "shall", "should",
        "may", "might", "must",
        "seem", "seems", "seemed",
        "appear", "appears", "appeared",
        "become", "becomes", "became",
        "get", "gets", "got",
        "remain", "remains", "remained",
    })

    @staticmethod
    def _clean(span: Any) -> str:
        return span.strip() if isinstance(span, str) else ""

    @staticmethod
    def _normalize(span: str) -> str:
        return _normalize_identity(span)

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 1 — Node extraction
    # ─────────────────────────────────────────────────────────────────────────

    def extract_nodes(self, annotation: Dict[str, Any]) -> Dict[str, List[Any]]:
        """
        Bucket the flat ``annotations`` list into typed groups.
        Now normalizes identity spans properly.
        """
        identities:  List[Dict[str, Any]] = []
        professions: List[str] = []
        triggers:    List[str] = []
        frames:      List[str] = []

        for ann in annotation.get("annotations", []):
            raw   = self._clean(ann.get("span", ""))
            if not raw:
                continue

            ntype = ann.get("node")

            # Only normalize identity nodes
            if ntype == self.IDENTITY_NODE:
                span = self._normalize(raw)
            else:
                span = raw.lower().strip()

            if ntype == self.IDENTITY_NODE:
                identities.append({
                    "span": span,
                    "label": ann.get("label"),
                    "category": _IDENTITY_TO_CATEGORY.get(span, "Other")
                })
            elif ntype == self.PROFESSION_NODE:
                professions.append(span)
            elif ntype == self.TRIGGER_NODE:
                triggers.append(span)
            elif ntype == self.FRAME_NODE:
                frames.append(span)

        return {
            "identities":  identities,
            "professions": professions,
            "triggers":    triggers,
            "frames":      frames,
        }

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 2 — Frame classification
    # ─────────────────────────────────────────────────────────────────────────

    def _is_copula_frame(self, frame: str) -> bool:
        """
        True if frame is a copula/auxiliary verb (meaningless on its own).
        These become edge labels instead of nodes.
        """
        tokens = frame.strip().lower()
        return bool(tokens) and tokens in self.PREDICATE_LEAD_WORDS

    # ─────────────────────────────────────────────────────────────────────────
    # STEP 3 — Relation extraction (UPDATED)
    # ─────────────────────────────────────────────────────────────────────────

    def extract_relations(self, nodes):

      identities = nodes["identities"]
      professions = nodes["professions"]
      triggers = nodes["triggers"]
      frames = nodes["frames"]

      relations = []

      # ----------------------------
      # Identity ↔ Identity
      # ----------------------------
      if len(identities) > 1:
          relations.extend(
              self._build_intersections(identities)
          )

      # ----------------------------
      # Identity ↔ Profession
      # ----------------------------
      if professions:
          relations.extend(
              self._build_role_associations(
                  identities,
                  professions
              )
          )

      # ----------------------------
      # Frame-driven patterns
      # ----------------------------
      if len(frames) == 1 and len(triggers) > 1:
        frame_pairs = [(t, frames[0]) for t in triggers]

      else:
        frame_pairs = list(zip(triggers, frames))

      for trigger, frame in frame_pairs:

          if self._is_copula_frame(frame):

              relations.extend(
                  self._build_predicate_relations(
                      identities,
                      professions,
                      trigger,
                      frame
                  )
              )

          else:

              relations.extend(
                  self._build_semantic_relations(
                      identities,
                      professions,
                      trigger,
                      frame
                  )
              )

      return self._dedupe(relations)

    # ── Relation builders ─────────────────────────────────────────────────────

    def _build_intersections(
        self, identities: List[Dict[str, Any]]
    ) -> List[Dict[str, str]]:
        return [
            self._edge(a["span"], b["span"], self.REL_IDENTITY_INTERSECTION)
            for a, b in itertools.combinations(identities, 2)
        ]

    def _build_role_associations(
        self,
        identities: List[Dict[str, Any]],
        professions: List[str],
    ) -> List[Dict[str, str]]:
        return [
            self._edge(i["span"], p, self.REL_IDENTITY_ROLE_ASSOCIATION)
            for i in identities
            for p in professions
        ]

    def _build_predicate_relations(self,identities,professions,trigger,frame):

      relations = []

      for identity in identities:

          relations.append(
              self._edge(
                  identity["span"],
                  trigger,
                  frame
              )
          )

      for profession in professions:

          relations.append(
              self._edge(
                  profession,
                  trigger,
                  frame
              )
          )

      return relations

    def _build_semantic_relations(self,identities, professions,trigger,frame):

        relations = []

        # Identity → Frame

        for identity in identities:

            relations.append(
                self._edge(
                    identity["span"],
                    frame,
                    self.REL_IDENTITY_FRAME_ASSOCIATION
                )
            )

        # Profession → Frame

        for profession in professions:

            relations.append(
                self._edge(
                    profession,
                    frame,
                    self.REL_IDENTITY_ROLE_ASSOCIATION
                )
            )

        # Frame → Trigger

        relations.append(
            self._edge(
                frame,
                trigger,
                self.REL_CHARACTERIZED_AS
            )
        )

        return relations


    # ── Utilities ─────────────────────────────────────────────────────────────

    @staticmethod
    def _edge(source: str, target: str, relation: str) -> Dict[str, str]:
        return {"source": source, "target": target, "relation": relation}

    @staticmethod
    def _dedupe(rels: List[Dict[str, str]]) -> List[Dict[str, str]]:
        seen: set = set()
        unique: List[Dict[str, str]] = []
        for r in rels:
            k = (r["source"], r["target"], r["relation"])
            if k not in seen:
                seen.add(k)
                unique.append(r)
        return unique

    # ── Public entry points (unchanged) ───────────────────────────────────

    def process_annotation(self,annotation: Dict[str, Any]) -> List[Dict[str, str]]:
      nodes = self.extract_nodes(annotation)
      return self.extract_relations(nodes)

    def process_batch(self, annotations: List[Dict[str, Any]]) -> List[Dict[str, str]]:
        out: List[Dict[str, str]] = []
        for ann in annotations:
            out.extend(self.process_annotation(ann))
        return out

In [9]:
# ── Per-sentence graph: typed nodes + safe handling of empty annotations ──────
# Each graph node is keyed by (normalised span, node type), so identical text
# assigned to different DER node types is preserved as distinct typed nodes.
# Empty annotations intentionally produce a sentence-only graph after add_sentence_node().

def _normalise_annotated_span(raw_span: str, node_type: str) -> str:
    raw = raw_span.strip()
    return _normalize_identity(raw) if node_type == "Identity Dimensions" else raw.lower().strip()


def _typed_node_id(span: str, node_type: str):
    return (span, node_type)


def _lookup_node_meta(span: str, raw_annotations: list) -> Tuple[str, Optional[str]]:
    """Compatibility helper: return the first matching annotation's type and label."""
    for ann in raw_annotations:
        raw = ann.get("span", "")
        if not isinstance(raw, str):
            continue
        node_type = ann.get("node", "Unknown")
        if _normalise_annotated_span(raw, node_type) == span:
            return node_type, ann.get("label")
    return "Unknown", None


def build_sentence_graph(
    annotation: Dict[str, Any],
    extractor: Optional[RuleBasedRelations] = None
) -> nx.DiGraph:
    if extractor is None:
        extractor = RuleBasedRelations()

    G = nx.DiGraph()
    raw_anns = annotation.get("annotations", []) or []

    # span -> all typed node IDs for that normalised text; keep duplicates by type.
    candidates = defaultdict(list)

    # 1. Add every valid annotation, including isolated nodes.
    for ann in raw_anns:
        raw_span = ann.get("span", "")
        if not isinstance(raw_span, str) or not raw_span.strip():
            continue
        node_type = ann.get("node", "Unknown")
        span = _normalise_annotated_span(raw_span, node_type)
        if not span:
            continue
        node_id = _typed_node_id(span, node_type)
        if node_id not in G:
            G.add_node(
                node_id,
                node_type=node_type,
                ontology_label=ann.get("label"),
                frequency=1,
                surface=raw_span.strip(),
            )
            candidates[span].append(node_id)
        else:
            G.nodes[node_id]["frequency"] += 1

    # Type expectations disambiguate relation endpoints that the current rule
    # extractor emits as plain text strings rather than typed node references.
    ID = RuleBasedRelations.IDENTITY_NODE
    PROF = RuleBasedRelations.PROFESSION_NODE
    TRIG = RuleBasedRelations.TRIGGER_NODE
    FRAME = RuleBasedRelations.FRAME_NODE
    expected_types = {
        RuleBasedRelations.REL_IDENTITY_INTERSECTION: ({ID}, {ID}),
        RuleBasedRelations.REL_CHARACTERIZED_AS: ({FRAME}, {TRIG}),
        RuleBasedRelations.REL_IDENTITY_FRAME_ASSOCIATION: ({ID}, {FRAME}),
        RuleBasedRelations.REL_IDENTITY_ROLE_ASSOCIATION: ({ID, PROF}, {PROF, FRAME}),
    }

    def resolve_endpoint(text, preferred_types=None):
        text = str(text).strip()
        matches = candidates.get(text, [])
        if preferred_types:
            preferred = [n for n in matches if G.nodes[n].get("node_type") in preferred_types]
            if preferred:
                matches = preferred
        if matches:
            return matches[0]
        # Preserve any unexpected relation endpoint as an Unknown typed node.
        node_id = _typed_node_id(text, "Unknown")
        if node_id not in G:
            G.add_node(node_id, node_type="Unknown", ontology_label=None,
                       frequency=1, surface=text)
        candidates[text].append(node_id)
        return node_id

    # 2. Add extracted relations, mapped onto type-aware node IDs.
    for rel in extractor.process_annotation(annotation):
        source_text, target_text = rel["source"], rel["target"]
        relation = rel["relation"]
        src_types, tgt_types = expected_types.get(relation, ({ID, PROF}, {TRIG}))
        source = resolve_endpoint(source_text, src_types)
        target = resolve_endpoint(target_text, tgt_types)
        if G.has_edge(source, target):
            G[source][target]["weight"] += 1
        else:
            G.add_edge(source, target, relation=relation, weight=1)

    G.graph["sentence"] = annotation.get("sentence", "")
    return G


In [10]:
SENTENCE_NODE_TYPE = "Sentence"
SENTENCE_RELATION = "contains"


def add_sentence_node(G):
    """
    Add a dedicated Sentence node to an existing
    sentence-level NetworkX graph.
    """

    G = G.copy()

    sentence = G.graph.get("sentence", "")

    # Unique node name for this graph
    sentence_node = "__SENTENCE__"

    # Add sentence node
    G.add_node(
        sentence_node,
        node_type=SENTENCE_NODE_TYPE,
        ontology_label=None,
        frequency=1,
        text=sentence
    )

    # Connect Sentence -> every existing semantic node
    for node in list(G.nodes()):

        if node == sentence_node:
            continue

        G.add_edge(
            sentence_node,
            node,
            relation=SENTENCE_RELATION,
            weight=1
        )

    return G

In [11]:
# Remember each graph node's original surface span (needed for contextual BERT alignment).
IDENTITY_NODE_NAME = "Identity Dimensions"

def _raw_span_for_graph_node(node, node_attrs, annotation):
    """Return the stored original span; fall back to annotation matching for older graphs."""
    if node_attrs.get("surface"):
        return str(node_attrs["surface"])
    ntype = node_attrs.get("node_type", "Unknown")
    node_text = node[0] if isinstance(node, tuple) and len(node) == 2 else str(node)
    for ann in annotation.get("annotations", []):
        if ann.get("node", "Unknown") != ntype:
            continue
        raw = str(ann.get("span", "")).strip()
        if not raw:
            continue
        cand = _normalize_identity(raw) if ntype == IDENTITY_NODE_NAME else raw.lower().strip()
        if cand == str(node_text).lower().strip():
            return raw
    return str(node_text)


def attach_surfaces(G, annotation):
    for n, a in G.nodes(data=True):
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            continue
        a["surface"] = _raw_span_for_graph_node(n, a, annotation)
    return G


In [12]:
IDENTITY_NODE = RuleBasedRelations.IDENTITY_NODE
CANONICAL_RELATIONS = {
    RuleBasedRelations.REL_IDENTITY_INTERSECTION,
    RuleBasedRelations.REL_IDENTITY_ROLE_ASSOCIATION,
    RuleBasedRelations.REL_CHARACTERIZED_AS,
    RuleBasedRelations.REL_IDENTITY_FRAME_ASSOCIATION,
    SENTENCE_RELATION,
}


def relation_family(rel: str) -> str:
    """Collapse the open-ended relation vocabulary (copula words such as 'is', 'can', ...)
    into 'predicate'; canonical DER relations and 'contains' are kept. Strips 'rev_'."""
    r = rel[4:] if rel.startswith("rev_") else rel
    return r if r in CANONICAL_RELATIONS else "predicate"


# ── NetworkX-level graph edits (used by ablations / counterfactuals / occlusion) ──
def nx_remove_der_edges(G, families=None):
    """Remove DER edges (everything except Sentence->node 'contains').
    families=None removes all; otherwise only edges whose relation_family is in `families`."""
    G2 = G.copy()
    drop = [(u, v) for u, v, a in G2.edges(data=True)
            if a["relation"] != SENTENCE_RELATION
            and (families is None or relation_family(a["relation"]) in families)]
    G2.remove_edges_from(drop)
    return G2


def nx_remove_node_type(G, node_type):
    G2 = G.copy()
    G2.remove_nodes_from([n for n, a in G2.nodes(data=True) if a.get("node_type") == node_type])
    return G2


# ── Build graphs for each dataset ───────────────────────────────────────────
extractor = RuleBasedRelations()


def build_dataset(annos_pos, annos_neg):
    annos = list(annos_pos) + list(annos_neg)
    labels = np.array([1] * len(annos_pos) + [0] * len(annos_neg))
    graphs = [attach_surfaces(add_sentence_node(build_sentence_graph(a, extractor)), a) for a in annos]
    # Groups: same template_id => same group (pair-safe CV). Items without one get their own group.
    groups = np.array([a["template_id"] if a.get("template_id") is not None else -(i + 1)
                       for i, a in enumerate(annos)])
    return dict(annos=annos, graphs=graphs, labels=labels, groups=groups)


DATA = {name: build_dataset(d["pos"], d["neg"]) for name, d in DATASETS.items()}
for name, d in DATA.items():
    n_nodes = np.mean([G.number_of_nodes() - 1 for G in d["graphs"]])
    n_der = np.mean([sum(a["relation"] != SENTENCE_RELATION for _, _, a in G.edges(data=True))
                     for G in d["graphs"]])
    print(f"{name}: {len(d['graphs'])} graphs | avg entity nodes {n_nodes:.1f} | avg DER edges {n_der:.1f}"
          f" | groups {len(set(d['groups']))}")

InDiBias: 326 graphs | avg entity nodes 2.8 | avg DER edges 1.6 | groups 326
IndiCasa: 326 graphs | avg entity nodes 5.8 | avg DER edges 4.7 | groups 326


In [13]:
# ── Graph coverage audit (run after DATA is built) ────────────────────────────
# Empty annotations are expected to have zero DER/entity nodes; they are not
# silently repaired or assigned invented relationships.
for name, d in DATA.items():
    rows = []
    for i, (ann, G, y) in enumerate(zip(d["annos"], d["graphs"], d["labels"])):
        entity_nodes = [(n, a) for n, a in G.nodes(data=True)
                        if a.get("node_type") != SENTENCE_NODE_TYPE]
        der_edges = [(u, v, a) for u, v, a in G.edges(data=True)
                     if a.get("relation") != SENTENCE_RELATION]
        if not ann.get("annotations") or not entity_nodes:
            rows.append({
                "dataset": name, "index": i, "label": int(y),
                "sentence": ann.get("sentence", ""),
                "annotation_count": len(ann.get("annotations", []) or []),
                "entity_node_count": len(entity_nodes),
                "der_edge_count": len(der_edges),
                "empty_annotations": not bool(ann.get("annotations")),
            })
    print(f"{name}: {len(rows)} records with empty annotations or no entity nodes")
    if rows:
        display(pd.DataFrame(rows).head(10))


InDiBias: 40 records with empty annotations or no entity nodes


,dataset,index,label,sentence,annotation_count,entity_node_count,der_edge_count,empty_annotations
0,InDiBias,27,1,They were asked to be careful while walking th...,0,0,0,True
1,InDiBias,73,1,Prakash learned to play chess very quickly and...,0,0,0,True
2,InDiBias,74,1,Anjali was terrified of bugs.,0,0,0,True
3,InDiBias,78,1,Kabir will not let his sister go for buying mi...,0,0,0,True
4,InDiBias,81,1,Saira served food in a restaurant.,0,0,0,True
5,InDiBias,86,1,Imran was the fastest on the draw and shot roh...,0,0,0,True
6,InDiBias,87,1,Raj took off his ring and put it in his pocket...,0,0,0,True
7,InDiBias,88,1,Raj took his wedding ring off so it wouldn't g...,0,0,0,True
8,InDiBias,89,1,Sunil spent most of his time writing programs ...,0,0,0,True
9,InDiBias,96,1,i'm glad i wasn't there since i would have bee...,0,0,0,True


IndiCasa: 1 records with empty annotations or no entity nodes


,dataset,index,label,sentence,annotation_count,entity_node_count,der_edge_count,empty_annotations
0,IndiCasa,108,1,The brother will not let his sister go for buy...,0,0,0,True


## 3. Frozen text encoders and baseline features

| Encoder | Sentence vector | What it tests |
|---|---|---|
| `SBERT` | `all-MiniLM-L6-v2` embedding of the full sentence (384d) | compact sentence-level representation |
| `BERT` | `bert-base-uncased` `[CLS]` vector for the full sentence (768d) | standard transformer sentence representation |
| `BERT-ISO` | used only for independently encoded DER-node text where needed by the preprocessing utilities | isolated node text representation |

All pretrained encoders are frozen. The sentence-only MLP baselines classify from the full-sentence vector; feature-based baselines use explicit counts of DER node types, node labels, relation families, or identity frequencies. This notebook does not train a graph neural network.


In [14]:
# ── graph-attribute helpers ────────────────────────────────────────────────
def sentence_node_name(G):
    return next(n for n, a in G.nodes(data=True) if a.get("node_type") == SENTENCE_NODE_TYPE)


def sentence_text(G):
    a = G.nodes[sentence_node_name(G)]
    return a.get("text", G.graph.get("sentence", ""))


def node_surface(G, n):          # original span text (falls back to the node name)
    return G.nodes[n].get("surface", str(n))


def node_ctx(G, n):              # sentence the node is contextualised in (counterfactual node-only swaps override this)
    return G.nodes[n].get("ctx", G.graph.get("sentence", ""))


def find_char_span(sentence, span_text):
    """Case-insensitive, word-bounded span lookup (handles simple plurals)."""
    if not sentence or not span_text:
        return None
    cands = [span_text]
    if span_text.lower().endswith("y"):
        cands.append(span_text[:-1] + "ies")
    cands += [span_text + "s", span_text + "es"]
    for c in cands:
        m = re.search(r"(?<!\w)" + re.escape(c) + r"(?!\w)", sentence, flags=re.IGNORECASE)
        if m:
            return m.start(), m.end()
    m = re.search(re.escape(span_text), sentence, flags=re.IGNORECASE)   # last resort
    return (m.start(), m.end()) if m else None


# ── featurisers ────────────────────────────────────────────────────────────
class Featurizer:
    """text/span -> vector cache. keys are hashable; vectors live on CPU in fp32."""
    name, dim = "?", 0

    def __init__(self):
        self.cache, self.failures, self.released = {}, set(), False

    def node_key(self, G, n):
        raise NotImplementedError

    def _compute(self, missing):
        raise NotImplementedError

    def keys(self, G, nodes=None):
        nodes = list(G.nodes) if nodes is None else nodes
        return [self.node_key(G, n) for n in nodes]

    def ensure(self, keys):
        missing = list(dict.fromkeys(k for k in keys if k not in self.cache))
        if not missing:
            return
        if self.released:
            raise RuntimeError(f"{self.name} encoder was released but {len(missing)} uncached keys were requested "
                               f"(e.g. {missing[0]}). Add those graphs to the precompute step in section 5.")
        self._compute(missing)

    def warm(self, graphs):
        self.ensure([k for G in graphs for k in self.keys(G)])

    def vectors(self, G, nodes):
        ks = self.keys(G, nodes)
        self.ensure(ks)
        return torch.stack([self.cache[k] for k in ks])

    def sentence_vector(self, G):
        k = self.node_key(G, sentence_node_name(G))
        self.ensure([k])
        return self.cache[k]


class SBERTFeaturizer(Featurizer):
    name = "SBERT"

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        super().__init__()
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name, device=str(device))
        if ENCODER_FP16 and device.type == "cuda":
            self.model.half()
        self.dim = int(self.model.get_sentence_embedding_dimension())
        self.model_name = model_name

    def node_key(self, G, n):
        a = G.nodes[n]
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            return ("t", a.get("text", G.graph.get("sentence", "")))
        return ("t", str(n))                       # node text in isolation (as in the original notebook)

    def _compute(self, missing):
        texts = [k[1] for k in missing]
        with torch.no_grad():
            embs = self.model.encode(texts, batch_size=ENC_BATCH, convert_to_tensor=True, show_progress_bar=False)
        embs = embs.float().cpu()
        for k, e in zip(missing, embs):
            self.cache[k] = e.clone()

    def release(self):
        self.model = None; self.released = True; free_gpu()


class BERTFeaturizer(Featurizer):
    name = "BERT"

    def __init__(self, model_name="bert-base-uncased"):
        super().__init__()
        from transformers import AutoTokenizer, AutoModel
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(model_name).to(device).eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.dim = int(self.model.config.hidden_size)
        self.model_name = model_name
        self.fp16 = bool(ENCODER_FP16 and device.type == "cuda")

    def node_key(self, G, n):
        a = G.nodes[n]
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            return ("cls", a.get("text", G.graph.get("sentence", "")))
        return ("span", node_ctx(G, n), node_surface(G, n))

    def _compute(self, missing):
        by_sent = defaultdict(list)
        for k in missing:
            by_sent[k[1]].append(k)
        sents = sorted(by_sent, key=len)          # length-sorted batches => little padding
        for b in range(0, len(sents), ENC_BATCH):
            chunk = sents[b:b + ENC_BATCH]
            enc = self.tok(chunk, return_tensors="pt", padding=True, truncation=True,
                           max_length=256, return_offsets_mapping=True)
            offsets = enc.pop("offset_mapping").numpy()
            enc = {k: v.to(device) for k, v in enc.items()}
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.fp16):
                hidden = self.model(**enc).last_hidden_state
            hidden = hidden.float().cpu()          # [B, T, 768] -> CPU immediately
            for i, s in enumerate(chunk):
                for k in by_sent[s]:
                    if k[0] == "cls":
                        vec = hidden[i, 0]
                    else:
                        span = find_char_span(s, k[2])
                        ids = [] if span is None else [j for j, (a, c) in enumerate(offsets[i])
                                                       if c > a and a < span[1] and c > span[0]]
                        if ids:
                            vec = hidden[i, ids].mean(dim=0)
                        else:
                            vec = torch.zeros(self.dim); self.failures.add((s, k[2]))
                    self.cache[k] = vec.clone()
            del hidden, enc

    def release(self):
        self.model = None; self.tok = None; self.released = True; free_gpu()


class BERTIsolatedFeaturizer(BERTFeaturizer):
    """BERT CLS vectors for each DER node's text in isolation; sentence nodes stay contextual CLS."""
    name = "BERT-ISO"

    def node_key(self, G, n):
        a = G.nodes[n]
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            return ("cls", a.get("text", G.graph.get("sentence", "")))
        return ("isolated", node_surface(G, n))

    def _compute(self, missing):
        # Sentence vectors use full-sentence CLS. DER nodes use standalone node text CLS.
        sentence_keys = [k for k in missing if k[0] == "cls"]
        isolated_keys = [k for k in missing if k[0] == "isolated"]
        batches = [(sentence_keys, [k[1] for k in sentence_keys]),
                   (isolated_keys, [k[1] for k in isolated_keys])]
        for keys, texts in batches:
            if not keys:
                continue
            for start in range(0, len(keys), ENC_BATCH):
                key_chunk = keys[start:start + ENC_BATCH]
                text_chunk = texts[start:start + ENC_BATCH]
                enc = self.tok(text_chunk, return_tensors="pt", padding=True,
                               truncation=True, max_length=256)
                enc = {k: v.to(device) for k, v in enc.items()}
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.fp16):
                    hidden = self.model(**enc).last_hidden_state[:, 0, :]
                hidden = hidden.float().cpu()
                for k, vec in zip(key_chunk, hidden):
                    self.cache[k] = vec.clone()
                del hidden, enc


FEATURIZERS: Dict[str, Featurizer] = {}


## 4. Baseline models and variant registry

Every model predicts the probability that a sentence is labelled stereotype (1).

| Family | Variants |
|---|---|
| Feature-based baselines | Identity prior · LR frames/triggers only · LR bag-of-types · LR bag-of-nodes |
| Leak control | LR bag-of-nodes (shuffled labels): same features and model as LR bag-of-nodes, trained on randomly permuted training labels and scored against the true labels |
| Sentence MLP | SBERT and BERT `[CLS]`, base/reg configurations |

The feature-based baselines use handcrafted counts extracted from the annotated sentence graph; they do not perform learned message passing. The sentence MLP baselines use only the frozen full-sentence embeddings.


In [15]:
class SentenceMLP(nn.Module):
    """Sentence-vector-only baseline (no graph)."""
    def __init__(self, input_dim=384, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden_dim, 2))

    def forward(self, x):
        return self.net(x)


In [16]:
# ═════════ shared helpers ═════════
def identity_names(G):
    return [n for n, a in G.nodes(data=True) if a["node_type"] == IDENTITY_NODE]


def graph_features(G, mode):
    f = Counter()
    if mode == "types":
        for _, a in G.nodes(data=True):
            f["nt::" + a["node_type"]] += 1
        for _, _, a in G.edges(data=True):
            f["rel::" + relation_family(a["relation"])] += 1
    else:
        for n, a in G.nodes(data=True):
            if a["node_type"] == SENTENCE_NODE_TYPE:
                continue
            if mode == "nodes_no_identity" and a["node_type"] == IDENTITY_NODE:
                continue
            f[a["node_type"] + "::" + str(n)] += 1
    return f


def _net_device(net):
    return next(net.parameters()).device




# ═════════ fitted-model wrappers: every baseline predicts P(class) for a LIST OF NETWORKX GRAPHS ═════════
class FittedModel:
    transform = None            # nx-level ablation transform applied before featurising (train AND test)

    def _prep(self, graphs):
        return [self.transform(G) for G in graphs] if self.transform else list(graphs)

    def predict_proba(self, graphs, zero_sentence=False):
        raise NotImplementedError

    def p_stereotype(self, graphs, zero_sentence=False):
        return self.predict_proba(graphs, zero_sentence)[:, 1] if len(graphs) else np.zeros(0)

    def to(self, dev):
        return self

    def offload(self):
        return self

class MLPFitted(FittedModel):
    def __init__(self, net, feat, transform=None, zero_sentence=False):
        self.net, self.feat, self.transform, self.zero_sentence = net, feat, transform, zero_sentence

    @torch.no_grad()
    def predict_proba(self, graphs, zero_sentence=False):
        zs = self.zero_sentence or zero_sentence
        X = torch.stack([self.feat.sentence_vector(G) for G in self._prep(graphs)])
        if zs:
            X = torch.zeros_like(X)
        self.net.eval()
        return F.softmax(self.net(X.to(_net_device(self.net))), dim=1).cpu().numpy()

    def to(self, dev):
        self.net.to(dev); return self

    def offload(self):
        self.net.to("cpu"); return self


class LRFitted(FittedModel):
    def __init__(self, dv, clf, mode, transform=None):
        self.dv, self.clf, self.mode, self.transform = dv, clf, mode, transform

    def predict_proba(self, graphs, zero_sentence=False):      # no sentence feature => zero_sentence is a no-op
        X = self.dv.transform([graph_features(G, self.mode) for G in self._prep(graphs)])
        return self.clf.predict_proba(X)


class ConstFitted(FittedModel):
    """Fallback when an ablation leaves a feature-based model with no features at all."""
    def __init__(self, p1, transform=None):
        self.p1, self.transform = float(p1), transform

    def predict_proba(self, graphs, zero_sentence=False):
        return np.tile([1 - self.p1, self.p1], (len(graphs), 1))


class IdentityPriorFitted(FittedModel):
    def __init__(self, counts, transform=None):
        self.counts, self.transform = counts, transform

    def predict_proba(self, graphs, zero_sentence=False):
        out = []
        for G in self._prep(graphs):
            ps = [(self.counts[n][1] + 1) / (sum(self.counts[n]) + 2) for n in identity_names(G) if n in self.counts]
            p1 = float(np.mean(ps)) if ps else 0.5
            out.append([1 - p1, p1])
        return np.array(out)


# ═════════ training functions ═══════
def fit_mlp(X, y, cfg, seed, input_dim):
    set_seed(seed)
    net = SentenceMLP(input_dim, cfg["hidden"], cfg["dropout"]).to(train_device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    Xd, yd = X.to(train_device), torch.as_tensor(y, dtype=torch.long, device=train_device)
    for _ in range(cfg["epochs"]):
        net.train(); opt.zero_grad()
        F.cross_entropy(net(Xd), yd).backward(); opt.step()
    return net


def fit_lr(graphs_t, y, mode, transform, C=1.0):
    dv = DictVectorizer()
    X = dv.fit_transform([graph_features(G, mode) for G in graphs_t])
    if X.shape[1] == 0:
        return ConstFitted(np.mean(y), transform)
    return LRFitted(dv, LogisticRegression(C=C, max_iter=2000).fit(X, y), mode, transform)


def fit_idprior(graphs_t, y, transform):
    c = defaultdict(lambda: [0, 0])
    for G, yi in zip(graphs_t, y):
        for n in identity_names(G):
            c[n][int(yi)] += 1
    return IdentityPriorFitted(dict(c), transform)


# ═════════ the variant registry ═════════
@dataclass(frozen=True)
class Variant:
    name: str
    kind: str                       # idprior | lr_types | lr_nodes | lr_nodes_noid | lr_nodes_shuf | mlp
    family: str                     # Baseline | Control | Sentence MLP
    feats: Optional[str] = None     # SBERT | BERT | None
    cfg_tag: Optional[str] = None   # base | reg

    @property
    def cfg(self):
        return {"base": BASE_CFG, "reg": REG_CFG}[self.cfg_tag]


VARIANTS: Dict[str, Variant] = {}
def _add(v): VARIANTS[v.name] = v

_add(Variant("Identity prior", "idprior", "Baseline"))
_add(Variant("LR frames/triggers only", "lr_nodes_noid", "Baseline"))
_add(Variant("LR bag-of-types", "lr_types", "Baseline"))
_add(Variant("LR bag-of-nodes", "lr_nodes", "Baseline"))
_add(Variant("LR bag-of-nodes (shuffled labels)", "lr_nodes_shuf", "Control"))   # leak control: trained on permuted labels
for _f, _lab in [("SBERT", "SBERT"), ("BERT", "BERT-CLS")]:
    for _tag in ("base", "reg"):
        _add(Variant(f"MLP {_lab} ({_tag})", "mlp", "Sentence MLP", _f, _tag))

N_VARIANTS_EXPECTED = 9      # 4 baselines + 1 shuffled-label control + 4 sentence MLPs
assert len(VARIANTS) == N_VARIANTS_EXPECTED, f"expected {N_VARIANTS_EXPECTED} variants, registry has {len(VARIANTS)}"
print(len(VARIANTS), "variants:"); [print("  -", n) for n in VARIANTS]


# ═════════ method factory: (variant, dataset, optional ablation) -> fit_predict(train_idx, test_idx, seed) ═════════
GCACHE = {}

def get_graphs(name, transform, tname):
    key = (name, tname)
    if key not in GCACHE:
        GCACHE[key] = [transform(G) if transform else G for G in DATA[name]["graphs"]]
    return GCACHE[key]

def purge_cache(tname):
    for k in [k for k in GCACHE if k[1] == tname]:
        del GCACHE[k]


class Keeper:
    """Keeps the first `max_n` fitted models (= repetition 0), parked on CPU, for the interpretability stage."""
    def __init__(self, max_n):
        self.items, self.max_n = [], max_n

    def add(self, model, te):
        if len(self.items) < self.max_n:
            self.items.append((model.offload(), np.array(te)))


def make_method(v, name, transform=None, tname="full", zero_sentence=False, keeper=None):
    d = DATA[name]; y = d["labels"]; G0 = d["graphs"]
    Gt = get_graphs(name, transform, tname)


    if v.kind == "mlp":
        feat = FEATURIZERS[v.feats]
        X = torch.stack([feat.sentence_vector(G) for G in Gt])
        if zero_sentence:
            X = torch.zeros_like(X)

        def fit_predict(tr, te, seed):
            net = fit_mlp(X[torch.as_tensor(tr)], y[tr], v.cfg, seed, feat.dim)
            model = MLPFitted(net, feat, transform, zero_sentence)
            probs = model.predict_proba([G0[i] for i in te])
            if keeper is not None:
                keeper.add(model, te)
            return probs
        return fit_predict

    mode = {"lr_types": "types", "lr_nodes": "nodes", "lr_nodes_noid": "nodes_no_identity", "lr_nodes_shuf": "nodes"}.get(v.kind)

    def fit_predict(tr, te, seed):
        if v.kind == "idprior":
            model = fit_idprior([Gt[i] for i in tr], y[tr], transform)
        else:
            y_fit = y[tr]
            if v.kind == "lr_nodes_shuf":          # leak control: permute TRAIN labels only; test labels stay true
                y_fit = np.random.default_rng(seed).permutation(y_fit)
            model = fit_lr([Gt[i] for i in tr], y_fit, mode, transform)
        probs = model.predict_proba([G0[i] for i in te])
        if keeper is not None:
            keeper.add(model, te)
        return probs
    return fit_predict


9 variants:
  - Identity prior
  - LR frames/triggers only
  - LR bag-of-types
  - LR bag-of-nodes
  - LR bag-of-nodes (shuffled labels)
  - MLP SBERT (base)
  - MLP SBERT (reg)
  - MLP BERT-CLS (base)
  - MLP BERT-CLS (reg)


## 5. Evaluation harness
Every method is `fit_predict(train_idx, test_idx, seed) -> P(class)`. `cross_validate` collects **out-of-fold** probabilities for all repetitions, so accuracy, macro-F1, log-loss and paired tests all come from one object. Splits are built once per (dataset, CV) and shared by all variants.


In [17]:
def make_splits(labels, groups, strategy, seeds, n_splits=5):
    reps = []
    for s in seeds:
        if strategy == "stratified":
            it = StratifiedKFold(n_splits, shuffle=True, random_state=s).split(np.zeros(len(labels)), labels)
        elif strategy == "grouped":
            it = StratifiedGroupKFold(n_splits, shuffle=True, random_state=s).split(np.zeros(len(labels)), labels, groups)
        else:
            raise ValueError(strategy)
        reps.append([(tr, te) for tr, te in it])
    return reps


def cross_validate(fit_predict, labels, reps, seeds):
    """Out-of-fold probabilities [n_repeats, N, 2]."""
    oof = np.zeros((len(reps), len(labels), 2))
    for r, (folds, s) in enumerate(zip(reps, seeds)):
        for f, (tr, te) in enumerate(folds, start=1):
            oof[r, te] = fit_predict(tr, te, s + f)
    return oof


def summarize(oof, labels, name):
    rows = []
    for r in range(oof.shape[0]):
        p = np.clip(oof[r], 1e-7, 1 - 1e-7)
        p = p / p.sum(1, keepdims=True)
        pred = p.argmax(1)
        rows.append((accuracy_score(labels, pred), f1_score(labels, pred, average="macro"),
                     log_loss(labels, p, labels=[0, 1])))
    a = np.array(rows)
    return {"method": name, "acc": a[:, 0].mean(), "acc_sd": a[:, 0].std(), "macro_f1": a[:, 1].mean(),
            "log_loss": a[:, 2].mean(), "n_repeats": oof.shape[0]}


def paired_bootstrap(oof_a, oof_b, labels, groups, n_boot=5000, seed=0):
    """Does A beat B? Bootstrap over *groups* (templates) so minimal pairs are resampled together."""
    ca = (oof_a.argmax(2) == labels).mean(0)
    cb = (oof_b.argmax(2) == labels).mean(0)
    diff = ca - cb
    ug = np.unique(groups)
    idx_by = {g: np.where(groups == g)[0] for g in ug}
    rng = np.random.default_rng(seed)
    boots = np.array([np.concatenate([diff[idx_by[g]] for g in rng.choice(ug, len(ug), replace=True)]).mean()
                      for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean()))
    return dict(acc_diff=diff.mean(), ci_low=lo, ci_high=hi, p_two_sided=p)


def holm(pvals):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj, run = np.empty(m), 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj


## 6. Perturbation set + one-time feature precompute (then free the encoders)

The counterfactual identity swaps are enumerated **once, independent of any model** (`CF[dataset][graph_idx]`): every model is evaluated on exactly the same list of (graph, old → new) swaps, in two modes:

- **sentence+node**: identity node *and* sentence text are rewritten (full counterfactual).
- **node-only**: only the identity node's representation changes; the Sentence node and all other nodes keep their original representation (for contextual BERT, the swapped node is contextualised in the rewritten sentence; isolated-text encoders encode the swapped node text independently).

Only swaps whose old identity actually occurs in the sentence text are kept, so the two modes always compare the same items. Then **every** text/span that any model, ablation or perturbation will ever need is encoded in one batched pass, and the encoders are released from the GPU.


In [21]:
_ACRONYMS = {"obc", "sc", "st", "sc/st", "scs", "sts"}


def swap_identity(G, old, new):
    """Return (G_full, G_node) for replacing identity node `old` by `new`, or None if not possible.
    `old` / `new` are identity TEXTS (e.g. "brahmin"); graph nodes are typed ids (text, node_type)."""
    old_id, new_id = (old, IDENTITY_NODE), (new, IDENTITY_NODE)
    if old_id not in G or new_id in G:
        return None
    text = G.graph.get("sentence", "")
    pat = re.compile(rf"\b{re.escape(old)}(s|es)?\b", re.IGNORECASE)
    if not pat.search(text):
        return None

    def repl(m):
        disp = new.upper() if new in _ACRONYMS else new
        if new not in _ACRONYMS and m.group(0)[:1].isupper():
            disp = disp[:1].upper() + disp[1:]
        return disp + (m.group(1) or "")
    new_text = pat.sub(repl, text)

    G_full = nx.relabel_nodes(G, {old_id: new_id}, copy=True)
    G_full.graph["sentence"] = new_text
    for n, a in G_full.nodes(data=True):
        if a.get("node_type") == SENTENCE_NODE_TYPE:
            a["text"] = new_text
    G_full.nodes[new_id]["surface"] = new

    G_node = nx.relabel_nodes(G, {old_id: new_id}, copy=True)
    for n, a in G_node.nodes(data=True):
        if a.get("node_type") != SENTENCE_NODE_TYPE:
            a["ctx"] = text                      # everything keeps its ORIGINAL context ...
    G_node.nodes[new_id]["ctx"] = new_text       # ... except the swapped identity node
    G_node.nodes[new_id]["surface"] = new
    return G_full, G_node


def build_counterfactual_set(name, min_count=2):
    graphs = DATA[name]["graphs"]
    counts = Counter(n[0] for G in graphs for n in identity_names(G))      # identity TEXT counts
    vocab = defaultdict(set)
    for t, c in counts.items():
        if c >= min_count:
            vocab[_IDENTITY_TO_CATEGORY.get(t, "Other")].add(t)
    out = defaultdict(list)
    for i, G in enumerate(graphs):
        for old_id in identity_names(G):
            old = old_id[0]
            cat = _IDENTITY_TO_CATEGORY.get(old, "Other")
            for new in sorted(vocab[cat] - {old}):
                res = swap_identity(G, old, new)
                if res is not None:
                    out[i].append(dict(old=old, new=new, category=cat, G_full=res[0], G_node=res[1]))
    return dict(out)


# Editable: which identities count as "privileged" vs "marginalised" for the direction summary
PRIVILEGED = {"brahmin", "kshatriya", "vaishya"}
MARGINALISED = {"obc", "sc/st", "sc", "st", "dalit", "adivasi", "shudra"}

def direction(old, new):
    if old in PRIVILEGED and new in MARGINALISED:
        return "privileged -> marginalised"
    if old in MARGINALISED and new in PRIVILEGED:
        return "marginalised -> privileged"
    return "other"


CF = {name: build_counterfactual_set(name) for name in sorted({p[0] for p in ANALYSIS_PLAN})}
for name, cfs in CF.items():
    print(f"{name}: {sum(len(v) for v in cfs.values())} counterfactual swaps over {len(cfs)} graphs "
          f"(identical for every model)")

# ── encode everything once, per encoder; free each encoder immediately afterwards ──
all_graphs = [G for d in DATA.values() for G in d["graphs"]]
for cfs in CF.values():
    for lst in cfs.values():
        for c in lst:
            all_graphs += [c["G_full"], c["G_node"]]

need = sorted({v.feats for v in VARIANTS.values() if v.feats})
for fname in need:
    t0 = time.time()
    FEATURIZERS[fname] = (SBERTFeaturizer() if fname == "SBERT" else BERTIsolatedFeaturizer() if fname == "BERT-ISO" else BERTFeaturizer())
    gpu_report(f"{fname} loaded")
    FEATURIZERS[fname].warm(all_graphs)
    n_vec = len(FEATURIZERS[fname].cache)
    gpu_report(f"{fname} after encoding {n_vec} vectors")
    FEATURIZERS[fname].release()
    gpu_report(f"{fname} released")
    print(f"  {fname}: dim={FEATURIZERS[fname].dim}, {n_vec} cached vectors, {time.time()-t0:.0f}s")

# ── BERT alignment report (restricted to the real dataset graphs, not the swapped ones) ──
if "BERT" in FEATURIZERS:
    bf = FEATURIZERS["BERT"]
    rows = []
    for name, d in DATA.items():
        for i, G in enumerate(d["graphs"]):
            for n, a in G.nodes(data=True):
                if a.get("node_type") != SENTENCE_NODE_TYPE and (node_ctx(G, n), node_surface(G, n)) in bf.failures:
                    rows.append(dict(dataset=name, idx=i, node=n, span=node_surface(G, n), node_type=a["node_type"],
                                     sentence=G.graph["sentence"]))
    BERT_ALIGNMENT = pd.DataFrame(rows, columns=["dataset", "idx", "node", "span", "node_type", "sentence"])
    tot = {name: sum(G.number_of_nodes() - 1 for G in d["graphs"]) for name, d in DATA.items()}
    print("\nBERT span-alignment failures (zero vector used):",
          {name: f"{int((BERT_ALIGNMENT.dataset == name).sum())}/{tot[name]} nodes" for name in DATA})
    if len(BERT_ALIGNMENT):
        display(BERT_ALIGNMENT.head(20))

InDiBias: 685 counterfactual swaps over 143 graphs (identical for every model)
IndiCasa: 1811 counterfactual swaps over 250 graphs (identical for every model)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[gpu] BERT loaded: running on CPU
[gpu] BERT after encoding 16049 vectors: running on CPU
[gpu] BERT released: running on CPU
  BERT: dim=768, 16049 cached vectors, 268s


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\Krish\AppData\Local\Temp\ipykernel_21708\2137093671.py:85: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = int(self.model.get_sentence_embedding_dimension())


[gpu] SBERT loaded: running on CPU
[gpu] SBERT after encoding 3775 vectors: running on CPU
[gpu] SBERT released: running on CPU
  SBERT: dim=384, 3775 cached vectors, 41s

BERT span-alignment failures (zero vector used): {'InDiBias': '0/904 nodes', 'IndiCasa': '0/1889 nodes'}


## 7. Main comparison — every variant, same splits
`RES[(dataset, cv)][variant]` holds out-of-fold probabilities `[repeats, N, 2]`. For plans in `ANALYSIS_PLAN`, the models of repetition 0 are kept (parked on CPU) so the interpretability stage explains **exactly the models whose accuracy is in the table**.


In [22]:
RES, SPL, KEEP = {}, {}, {}
FAILED = []
t_all = time.time()
for plan in EVAL_PLAN:
    name, cv = plan
    d = DATA[name]; y = d["labels"]
    SPL[plan] = make_splits(y, d["groups"], cv, SEEDS)
    RES[plan], KEEP[plan] = {}, {}
    for vname, v in VARIANTS.items():
        t0 = time.time()
        try:
            keeper = Keeper(max_n=len(SPL[plan][0])) if plan in ANALYSIS_PLAN else None
            RES[plan][vname] = cross_validate(make_method(v, name, keeper=keeper), y, SPL[plan], SEEDS)
        except Exception as e:                      # keep going so every other variant still shows up
            import traceback; traceback.print_exc()
            FAILED.append((name, cv, vname, repr(e)))
            print(f"!! {name}/{cv} {vname} FAILED: {e!r}")
            continue
        if keeper is not None:
            KEEP[plan][vname] = keeper.items
        s = summarize(RES[plan][vname], y, vname)
        print(f"{name:9s}/{cv:10s} {vname:36s} acc={s['acc']:.3f} f1={s['macro_f1']:.3f} ({time.time()-t0:.0f}s)")
    free_gpu()
    gpu_report(f"after {name}/{cv}")
done = sum(len(RES[p]) for p in EVAL_PLAN)
print(f"main comparison done in {(time.time()-t_all)/60:.1f} min | {done}/{len(VARIANTS) * len(EVAL_PLAN)} variant x plan runs completed")
if FAILED:
    raise RuntimeError("These variants failed (everything else is in RES):\n" + "\n".join(f"  {a}/{b}: {c}: {d}" for a, b, c, d in FAILED))

InDiBias /stratified Identity prior                       acc=0.472 f1=0.458 (0s)
InDiBias /stratified LR frames/triggers only              acc=0.824 f1=0.683 (0s)
InDiBias /stratified LR bag-of-types                      acc=0.898 f1=0.862 (0s)
InDiBias /stratified LR bag-of-nodes                      acc=0.830 f1=0.699 (0s)
InDiBias /stratified LR bag-of-nodes (shuffled labels)    acc=0.740 f1=0.447 (0s)
InDiBias /stratified MLP SBERT (base)                     acc=0.773 f1=0.591 (3s)
InDiBias /stratified MLP SBERT (reg)                      acc=0.749 f1=0.444 (1s)
InDiBias /stratified MLP BERT-CLS (base)                  acc=0.773 f1=0.642 (3s)
InDiBias /stratified MLP BERT-CLS (reg)                   acc=0.755 f1=0.492 (1s)
[gpu] after InDiBias/stratified: running on CPU
IndiCasa /stratified Identity prior                       acc=0.737 f1=0.581 (0s)
IndiCasa /stratified LR frames/triggers only              acc=0.805 f1=0.642 (0s)
IndiCasa /stratified LR bag-of-types              

In [23]:
rows = []
for plan, methods in RES.items():
    y = DATA[plan[0]]["labels"]
    for m, oof in methods.items():
        r = summarize(oof, y, m); v = VARIANTS[m]
        r.update(dataset=plan[0], cv=plan[1], family=v.family, features=v.feats or "-")
        rows.append(r)
results_df = pd.DataFrame(rows)[["dataset", "cv", "family", "features", "method", "acc", "acc_sd", "macro_f1", "log_loss", "n_repeats"]]
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
for plan in EVAL_PLAN:
    print(f"\n=== {plan[0]} / {plan[1]} CV ===")
    print(results_df[(results_df.dataset == plan[0]) & (results_df.cv == plan[1])]
          .drop(columns=["dataset", "cv"]).round(3).to_string(index=False))

# ── Leak check: real LR bag-of-nodes vs the same model trained on shuffled labels ──
_REAL, _SHUF = "LR bag-of-nodes", "LR bag-of-nodes (shuffled labels)"
print("\n=== Label-shuffle leak check (scored against the TRUE labels) ===")
for plan in EVAL_PLAN:
    name, cv = plan; y = DATA[name]["labels"]
    sub_ = results_df[(results_df.dataset == name) & (results_df.cv == cv)].set_index("method")
    if _REAL in sub_.index and _SHUF in sub_.index:
        maj = max(float(np.mean(y)), 1 - float(np.mean(y)))
        print(f"{name}/{cv}: real acc={sub_.loc[_REAL, 'acc']:.3f} macro-F1={sub_.loc[_REAL, 'macro_f1']:.3f} | "
              f"shuffled acc={sub_.loc[_SHUF, 'acc']:.3f} macro-F1={sub_.loc[_SHUF, 'macro_f1']:.3f} | majority-class acc={maj:.3f}")
print("Reading: with no label signal the shuffled control should sit near the majority-class accuracy and a low macro-F1. "
      "A shuffled score clearly above that would mean label information reaches the test features.")

# ── Pre-specified paired comparisons (same for every dataset/CV), Holm-corrected within each plan ──
def comparison_pairs():
    pairs = []
    for tag in ("base", "reg"):
        pairs.append(("BERT-CLS vs SBERT sentence-only MLP", f"MLP BERT-CLS ({tag})", f"MLP SBERT ({tag})"))
    pairs.append(("LR bag-of-nodes vs its label-shuffled control", _REAL, _SHUF))
    return pairs

SIG = []
for plan in EVAL_PLAN:
    name, cv = plan; d = DATA[name]
    rows = []
    for question, a, b in comparison_pairs():
        if a not in RES[plan] or b not in RES[plan]:
            continue
        r = paired_bootstrap(RES[plan][a], RES[plan][b], d["labels"], d["groups"], n_boot=N_BOOT)
        rows.append(dict(dataset=name, cv=cv, question=question, A=a, B=b, **r))
    if rows:
        df = pd.DataFrame(rows); df["p_holm"] = holm(df.p_two_sided.values)
        SIG.append(df)
sig_df = pd.concat(SIG, ignore_index=True) if SIG else pd.DataFrame()
for plan in EVAL_PLAN:
    show = sig_df[(sig_df.dataset == plan[0]) & (sig_df.cv == plan[1])].drop(columns=["dataset", "cv"]) if len(sig_df) else sig_df
    print(f"\n=== Paired bootstrap, {plan[0]} / {plan[1]} (acc_diff = A - B, CI over groups, Holm-adjusted p) ===")
    print(show.round(3).to_string(index=False))



=== InDiBias / stratified CV ===
      family features                            method   acc  acc_sd  macro_f1  log_loss  n_repeats
    Baseline        -                    Identity prior 0.472   0.003     0.458     0.627          3
    Baseline        -           LR frames/triggers only 0.824   0.001     0.683     0.401          3
    Baseline        -                   LR bag-of-types 0.898   0.004     0.862     0.232          3
    Baseline        -                   LR bag-of-nodes 0.830   0.004     0.699     0.386          3
     Control        - LR bag-of-nodes (shuffled labels) 0.740   0.004     0.447     0.609          3
Sentence MLP    SBERT                  MLP SBERT (base) 0.773   0.007     0.591     0.507          3
Sentence MLP    SBERT                   MLP SBERT (reg) 0.749   0.003     0.444     0.587          3
Sentence MLP     BERT               MLP BERT-CLS (base) 0.773   0.003     0.642     0.553          3
Sentence MLP     BERT                MLP BERT-CLS (reg) 0